# Crop Recommendation System - Togo Agro-climatic Zones

**Multi-class classification pipeline for smallholder crop advisory**

Crops: Maize, Cassava, Sorghum, Yam, Soybean  
Features: rainfall, temperature, soil NPK, pH, humidity, solar radiation, agro-climatic zone  
Model: Soft-voting ensemble (Random Forest + Gradient Boosting + Logistic Regression)  

---

## Section 1 - Dependencies

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
os.makedirs('figures', exist_ok=True)
print('Dependencies loaded.')

## Section 2 - Synthetic Data Generation

Parametric simulation of agro-climatic conditions across five Togolese regions.
Crop-specific soil and climate offsets are grounded in FAO and IITA agronomic literature.
Label noise of 3% simulates annotation error from field surveys.

In [ ]:
N = 2000
CROPS = ['Maize', 'Cassava', 'Sorghum', 'Yam', 'Soybean']

ZONE_PARAMS = {
    'Savane':   {'rainfall_mm': (750, 120),  'temp_mean_c': (29.5, 1.5), 'soil_ph': (6.2, 0.5),
                 'soil_n_ppm': (120, 30), 'soil_p_ppm': (22, 8), 'soil_k_ppm': (180, 40),
                 'humidity_pct': (52, 8), 'solar_rad': (21, 2),
                 'crop_prior': [0.35, 0.20, 0.25, 0.10, 0.10]},
    'Kara':     {'rainfall_mm': (1050, 130), 'temp_mean_c': (27.5, 1.8), 'soil_ph': (6.5, 0.4),
                 'soil_n_ppm': (145, 35), 'soil_p_ppm': (28, 9), 'soil_k_ppm': (210, 45),
                 'humidity_pct': (63, 9), 'solar_rad': (19, 2),
                 'crop_prior': [0.30, 0.25, 0.15, 0.15, 0.15]},
    'Centrale': {'rainfall_mm': (1100, 140), 'temp_mean_c': (27.0, 1.6), 'soil_ph': (6.6, 0.4),
                 'soil_n_ppm': (155, 35), 'soil_p_ppm': (30, 10), 'soil_k_ppm': (220, 50),
                 'humidity_pct': (67, 8), 'solar_rad': (18, 2),
                 'crop_prior': [0.30, 0.30, 0.10, 0.20, 0.10]},
    'Plateaux': {'rainfall_mm': (1300, 150), 'temp_mean_c': (25.5, 1.5), 'soil_ph': (6.8, 0.3),
                 'soil_n_ppm': (170, 40), 'soil_p_ppm': (35, 10), 'soil_k_ppm': (240, 50),
                 'humidity_pct': (74, 7), 'solar_rad': (17, 2),
                 'crop_prior': [0.25, 0.35, 0.05, 0.25, 0.10]},
    'Maritime': {'rainfall_mm': (1450, 160), 'temp_mean_c': (27.0, 1.4), 'soil_ph': (5.9, 0.5),
                 'soil_n_ppm': (160, 38), 'soil_p_ppm': (32, 10), 'soil_k_ppm': (230, 55),
                 'humidity_pct': (80, 6), 'solar_rad': (17, 2),
                 'crop_prior': [0.20, 0.40, 0.05, 0.20, 0.15]},
}

ZONES = list(ZONE_PARAMS.keys())
zone_counts = np.random.multinomial(N, [1/5]*5)

CROP_OFFSET = {
    'Maize':   {'rainfall_mm': 250,  'soil_n_ppm': 100, 'soil_p_ppm': 25,  'temp_mean_c': 2.0,  'soil_ph':  0.5},
    'Cassava': {'rainfall_mm': -200, 'soil_n_ppm': -70, 'soil_p_ppm': -18, 'temp_mean_c': 1.5,  'soil_ph': -0.6},
    'Sorghum': {'rainfall_mm': -380, 'soil_n_ppm': -90, 'soil_p_ppm': -20, 'temp_mean_c': 4.0,  'soil_ph':  0.2},
    'Yam':     {'rainfall_mm': 320,  'soil_n_ppm': 110, 'soil_p_ppm': 32,  'temp_mean_c': -2.0, 'soil_ph': -0.3},
    'Soybean': {'rainfall_mm': 150,  'soil_n_ppm': 140, 'soil_p_ppm': 45,  'temp_mean_c': -3.5, 'soil_ph':  0.7},
}

rows = []
for zone, count in zip(ZONES, zone_counts):
    p = ZONE_PARAMS[zone]
    for _ in range(count):
        crop = np.random.choice(CROPS, p=p['crop_prior'])
        off  = CROP_OFFSET[crop]
        rows.append({
            'zone':          zone,
            'rainfall_mm':   max(300, np.random.normal(p['rainfall_mm'][0] + off['rainfall_mm'], p['rainfall_mm'][1])),
            'temp_mean_c':   np.random.normal(p['temp_mean_c'][0] + off['temp_mean_c'], p['temp_mean_c'][1]),
            'soil_ph':       np.clip(np.random.normal(p['soil_ph'][0] + off['soil_ph'], p['soil_ph'][1]), 4.5, 8.0),
            'soil_n_ppm':    max(50, np.random.normal(p['soil_n_ppm'][0] + off['soil_n_ppm'], p['soil_n_ppm'][1])),
            'soil_p_ppm':    max(5,  np.random.normal(p['soil_p_ppm'][0] + off['soil_p_ppm'], p['soil_p_ppm'][1])),
            'soil_k_ppm':    max(80, np.random.normal(p['soil_k_ppm'][0], p['soil_k_ppm'][1])),
            'humidity_pct':  np.clip(np.random.normal(p['humidity_pct'][0], p['humidity_pct'][1]), 30, 99),
            'solar_rad_mj':  max(10, np.random.normal(p['solar_rad'][0], p['solar_rad'][1])),
            'label_noise':   np.random.rand(),
            'crop':          crop,
        })

df = pd.DataFrame(rows)
flip_idx = df[df['label_noise'] < 0.03].index
df.loc[flip_idx, 'crop'] = np.random.choice(CROPS, size=len(flip_idx))
df.drop(columns=['label_noise'], inplace=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['crop'])

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(train_df['crop'].value_counts())

## Section 3 - Exploratory Data Analysis

In [ ]:
COLORS = {'Maize': '#e6a817', 'Cassava': '#2e8b57', 'Sorghum': '#c0392b',
          'Yam': '#8e44ad', 'Soybean': '#2980b9'}

plot_features = [
    ('rainfall_mm',  'Rainfall (mm)'),
    ('temp_mean_c',  'Mean Temperature (C)'),
    ('soil_ph',      'Soil pH'),
    ('soil_n_ppm',   'Soil Nitrogen (ppm)'),
    ('soil_p_ppm',   'Soil Phosphorus (ppm)'),
    ('soil_k_ppm',   'Soil Potassium (ppm)'),
    ('humidity_pct', 'Humidity (%)'),
    ('solar_rad_mj', 'Solar Radiation (MJ/m2)'),
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('Agro-climatic Features by Crop', fontsize=13)

for idx, (feat, label) in enumerate(plot_features):
    ax = axes[idx // 3][idx % 3]
    for crop in CROPS:
        vals = train_df[train_df['crop'] == crop][feat]
        ax.hist(vals, bins=25, alpha=0.55, color=COLORS[crop], label=crop)
    ax.set_title(label, fontsize=9)
    ax.tick_params(labelsize=7)

ax = axes[2][2]
zone_crop = train_df.groupby(['zone', 'crop']).size().unstack(fill_value=0)
zone_crop.plot(kind='bar', ax=ax, color=list(COLORS.values()), legend=False, width=0.75)
ax.set_title('Crop distribution by zone', fontsize=9)
ax.tick_params(axis='x', labelsize=6, rotation=30)

handles = [plt.Rectangle((0,0),1,1, color=COLORS[c]) for c in CROPS]
fig.legend(handles, CROPS, loc='lower center', ncol=5, fontsize=8,
           frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('figures/01_eda_agro_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

## Section 4 - Preprocessing and Model Definition

In [ ]:
NUMERIC_FEATURES = ['rainfall_mm', 'temp_mean_c', 'soil_ph',
                    'soil_n_ppm', 'soil_p_ppm', 'soil_k_ppm',
                    'humidity_pct', 'solar_rad_mj']
CATEGORICAL_FEATURES = ['zone']
TARGET = 'crop'

X_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_test  = test_df[TARGET]

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
CLASSES = le.classes_

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])

rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=SEED, n_jobs=-1)
gb = GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=SEED)
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)

ensemble = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('lr', lr)],
    voting='soft'
)

pipeline = Pipeline([('prep', preprocessor), ('model', ensemble)])
print('Pipeline defined.')

## Section 5 - Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(pipeline, X_train, y_train_enc, cv=cv, scoring='accuracy', n_jobs=-1)
print(f'CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print(f'Per-fold: {[round(s, 4) for s in cv_scores]}')

## Section 6 - Test Evaluation and Classification Report

In [ ]:
pipeline.fit(X_train, y_train_enc)
y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test_enc, y_pred)
f1  = f1_score(y_test_enc, y_pred, average='weighted')

print(f'Test Accuracy      : {acc:.4f}')
print(f'Test F1 (weighted) : {f1:.4f}')
print()
print(classification_report(y_test_enc, y_pred, target_names=CLASSES))

## Section 7 - Results Dashboard

In [ ]:
rf_fitted     = pipeline.named_steps['model'].estimators_[0]
ohe_cols      = pipeline.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES)
feature_names = NUMERIC_FEATURES + list(ohe_cols)
importances   = rf_fitted.feature_importances_

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(range(1, 6), cv_scores, color='#2e86c1', edgecolor='white', linewidth=0.6)
ax1.axhline(cv_scores.mean(), color='#c0392b', linestyle='--', linewidth=1.2,
            label=f'Mean {cv_scores.mean():.3f}')
ax1.set_title('5-Fold CV Accuracy', fontsize=9)
ax1.set_ylim(0.75, 1.0)
ax1.legend(fontsize=7)
ax1.tick_params(labelsize=7)

ax2 = fig.add_subplot(gs[0, 1:])
top8_idx   = np.argsort(importances)[-8:]
top8_names = [feature_names[i].replace('zone_', 'zone:') for i in top8_idx]
ax2.barh(top8_names, importances[top8_idx], color='#1a5276', edgecolor='white', linewidth=0.5)
ax2.set_title('Top 8 Feature Importances (Random Forest)', fontsize=9)
ax2.set_xlabel('Mean Decrease in Impurity', fontsize=8)
ax2.tick_params(labelsize=7)

ax3 = fig.add_subplot(gs[1, 0:2])
cm   = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(ax=ax3, colorbar=False, cmap='Blues')
ax3.set_title('Confusion Matrix (test set)', fontsize=9)
for lbl in ax3.get_xticklabels(): lbl.set_rotation(25)

ax4 = fig.add_subplot(gs[1, 2])
per_class_f1 = f1_score(y_test_enc, y_pred, average=None)
bars = ax4.bar(CLASSES, per_class_f1, color=[COLORS[c] for c in CLASSES],
               edgecolor='white', linewidth=0.6)
ax4.set_title('Per-class F1 Score', fontsize=9)
ax4.set_ylim(0.70, 1.0)
ax4.tick_params(axis='x', labelsize=7, rotation=20)
for bar, val in zip(bars, per_class_f1):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.2f}', ha='center', va='bottom', fontsize=7)

metrics_text = (f'Summary Metrics\n'
                f'CV Accuracy : {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}\n'
                f'Test Accuracy: {acc:.3f}\n'
                f'Test F1 (weighted): {f1:.3f}\n'
                f'N train: {len(train_df)} | N test: {len(test_df)}')
fig.text(0.01, 0.98, metrics_text, va='top', ha='left', fontsize=8, family='monospace',
         bbox=dict(boxstyle='round,pad=0.4', fc='#eaf2f8', ec='#2e86c1', lw=0.8))

fig.suptitle('Crop Recommendation - Results Dashboard', fontsize=13, y=1.01)
plt.savefig('figures/02_results_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

## Section 8 - Inference Example

Single-sample prediction demonstrating the advisory use case.

In [ ]:
# Example: a field in Centrale region with medium rainfall and moderate soil fertility
sample = pd.DataFrame([{
    'rainfall_mm':  1100,
    'temp_mean_c':  27.5,
    'soil_ph':      6.5,
    'soil_n_ppm':   145,
    'soil_p_ppm':   28,
    'soil_k_ppm':   215,
    'humidity_pct': 66,
    'solar_rad_mj': 18.5,
    'zone':         'Centrale'
}])

probas = pipeline.predict_proba(sample)[0]
pred   = le.inverse_transform([np.argmax(probas)])[0]

print(f'Recommended crop : {pred}')
print('Probabilities by crop:')
for crop, prob in sorted(zip(CLASSES, probas), key=lambda x: -x[1]):
    bar = '#' * int(prob * 30)
    print(f'  {crop:<10} {prob:.3f}  {bar}')